# MemoryNPC Final NLP Assignment Notebook

This notebook is the consolidated notebook for my university NLP assignment. I use it to explain the design challenge, architecture, motivated choices, validation protocol, results, limitations, demo-video plan, and code appendices.

I designed MemoryNPC myself around the assignment brief. The implementation still lives in normal Python files so the Streamlit app can run cleanly, but the full explanation and source code are included here.

## 1. Design Challenge

**Design a LangChain-based NPC memory agent to enable players in narrative game contexts to interact with non-player characters that remember previous conversations, retrieve relevant memories, and respond with context-aware dialogue with at least 80% memory retrieval success on manually designed test cases.**

I chose context-aware NPC dialogue with external long-term memory as my language task. A normal LLM can roleplay a fantasy blacksmith, but it cannot reliably maintain inspectable memory, explain why a relationship state changed, or expose which previous facts grounded its answer.

## 2. Why This Is More Than A Prompt

I added task-specific NLP functionality around an LLM:

- intent detection classifies what the player is doing;
- memory extraction decides whether an utterance should become durable memory;
- embeddings and FAISS retrieve memories by semantic similarity;
- deterministic trust logic tracks social state;
- response generation is grounded in retrieved memories and current trust state;
- validation traces record the pipeline decisions for every turn;
- JSON state export/import persists NPC memory across sessions.

This means the LLM is one component inside a larger language system rather than the whole solution.

## 3. Architecture

```mermaid
flowchart TD
    A[Player message] --> B[Intent detection chain]
    A --> C[Memory extraction chain]
    C --> D[Durable memory filter]
    D --> E[FAISS vector store]
    A --> F[Semantic memory retrieval]
    E --> F
    B --> G[Deterministic trust update]
    F --> H[Grounded response prompt]
    G --> H
    I[Recent conversation] --> H
    H --> J[LLM response as Elara]
    J --> K[Validation trace]
    E --> L[JSON state export/import]
    G --> L
    K --> L
```

My main design choice is separation of concerns. I separated intent, memory extraction, retrieval, trust, and response generation so each part can be inspected and tested by a reader rather than hidden inside one large prompt.

## 4. Motivated Design Choices

**LangChain** is used because I chain multiple NLP steps instead of using a single prompt. Prompt templates and model chains keep the intent detector, memory extractor, and response generator separate.

**FAISS and embeddings** are used because memory questions are often semantic. A player can ask about a "weapon" even when the stored memory says "sword." Keyword overlap is included as a baseline and performs worse on semantic queries.

**Deterministic trust state** is used because I wanted relationship changes to be repeatable and explainable. Letting the LLM decide trust changes would make validation harder.

**A role-only LLM baseline** is included because it shows what disappears when the system is reduced to a normal chatbot: no external memory, no retrieval evidence, no trust metadata, no persistence, and no validation trace.

**OpenAI models** are used as the default implementation because they are stable and easy to access through LangChain for this assignment. The architecture is model-agnostic: a local Ollama model, Hugging Face model, sentence-transformers embeddings, or another LangChain-compatible provider could replace the default chat and embedding models.

## 5. Validation Strategy

I designed the validation to be evaluation-focused. Instead of only showing a nice chat transcript, I show the internal NLP decisions that justify the system design. This matters because open-ended dialogue cannot be judged only by whether a response sounds fluent.

I validate:

- intent accuracy on manually labelled examples;
- top-3 semantic retrieval success on manually designed memory questions;
- FAISS retrieval against a keyword-overlap baseline;
- deterministic trust arithmetic;
- advice-following and advice-ignoring behavior;
- unsupported-memory prompts to check hallucinated recall;
- long scripted conversations;
- validation trace completeness;
- lightweight automatic response-grounding checks.

The 100% scores are controlled results on manually designed validation cases. I do not present them as perfect general performance on all possible player language.

## 6. Executed Results

I produced the original notebook results and then added a larger extended validation suite. The extended suite tests every major metric at least 20 times.

| Metric | Cases | Result | Interpretation |
|---|---:|---:|---|
| Intent classification | 30 | 100% | My classifier passed the expanded closed-label intent cases. |
| Memory extraction selectivity | 22 | 100% | Durable facts were saved and filler was ignored after a validation-driven fix. |
| FAISS semantic retrieval | 25 | 100% | My vector memory met the 80% target on expanded retrieval cases. |
| Keyword retrieval baseline | 25 | 88% | Keyword overlap was weaker than semantic retrieval. |
| Trust rule validation | 22 | 100% | Deterministic trust arithmetic worked as expected. |
| Advice behavior validation | 22 | 100% | Following and ignoring advice changed trust correctly. |
| Persistence roundtrip | 20 | 100% | JSON state export/import preserved memory and retrieval. |
| Full-pipeline trace completeness | 20 | 100% | Generated turns logged the required validation fields. |
| Full-pipeline trust arithmetic | 20 | 100% | Logged trust deltas matched before/after scores. |
| Full-pipeline automatic response checks | 20 | 100% | Generated turns passed trace and grounding checks. |

The evidence is not only the high score. It also matters that I compare FAISS against a baseline, run each major metric at least 20 times, and expose intermediate decisions through the validation trace. The validation results are summarized in `validation.ipynb` and `validation.html`.

## 7. Trustworthiness Measures

I make the app more trustworthy by combining LLM behavior with inspectable controls:

- restricted intent labels prevent arbitrary labels;
- durable memory extraction filters out low-value small talk;
- retrieved memories are shown before response generation;
- the prompt tells the model not to claim memories unless they were retrieved;
- trust changes are deterministic and logged;
- every turn stores input, intent, memory extraction, retrieved memories, trust before/after, advice status, and response;
- memory state can be exported and loaded again;
- automatic validation checks trace completeness and simple grounding behavior.

This does not remove all LLM risk, but it makes failures easier for me and a reader to detect and explain.

## 8. Persistence

I added persistence so memory is not only runtime state. `MemoryNPC` supports JSON export/import of:

- trust score;
- turn number;
- durable memories;
- conversation history;
- validation event log;
- pending advice state.

When a saved state is loaded, the FAISS index is rebuilt from the stored memories. In the Streamlit app, this appears as **Download memory state** and **Load memory state** in the sidebar.

This is a simple persistence layer. A production game would likely store the same information in a database or game-save system.

## 9. How To Run

Clone the repository or download the project folder, then run these commands from the project root:

```bash
git clone <repository-url>
cd memorynpc_project
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env
```

On Windows PowerShell, activate the environment with:

```powershell
.\.venv\Scripts\Activate.ps1
```

Add a valid API key to `.env`:

```bash
OPENAI_API_KEY=your_api_key_here
OPENAI_MODEL=gpt-4o-mini
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
```

Run the app:

```bash
streamlit run app.py
```

## 10. Limitations

- My validation set is small and manually designed. Larger user-written and held-out test sets would be needed for broader claims.
- Response quality still partly requires human judgement because dialogue quality is open-ended.
- JSON persistence is enough for this assignment, but not a production game-save architecture.
- The default implementation requires an OpenAI API key, although the architecture can use other LangChain-compatible models.
- The trust model is intentionally simple and interpretable, not a full social simulation.
- The app is not integrated into a real game engine.

## 11. Conclusion

MemoryNPC satisfies the assignment because I created a novel language agent that an LLM cannot do out of the box: an NPC with inspectable long-term memory, semantic retrieval, deterministic relationship state, persistence, and validation traces.

It meets the target on manually designed validation cases and clearly motivates why each NLP component exists.

## Appendix: `npc_agent.py`

```python
"""Shared NLP backend for the MemoryNPC project.

This file contains the actual assignment logic. The Streamlit app is only a
demo surface; this class is where the NLP pipeline is built:

player text -> intent detection -> memory extraction -> FAISS memory storage
-> semantic retrieval -> deterministic trust update -> grounded response.

The comments are intentionally detailed because the project is graded on design
motivation as well as working code. They explain both what each block does and
why it was implemented this way.
"""

import json
import os
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


class MemoryNPC:
    """A small LangChain-powered NPC agent with vector memory and trust tracking."""

    # A closed label set makes intent detection testable. Without this list, the
    # LLM could invent labels such as "question" or "angry_request", which would
    # make trust updates and validation inconsistent.
    ALLOWED_INTENTS = {
        "greeting",
        "ask_quest",
        "share_fact",
        "ask_memory",
        "compliment",
        "insult",
        "ask_help",
        "goodbye",
        "unknown",
    }

    # Trust is deliberately deterministic instead of letting the LLM decide
    # relationship changes. This makes the social state inspectable, repeatable,
    # and easy to validate in the notebook.
    TRUST_RULES = {
        "compliment": 5,
        "insult": -10,
        "ask_help": 0,
        "ask_quest": 0,
        "share_fact": 1,
        "greeting": 1,
        "goodbye": 0,
        "unknown": 0,
        "ask_memory": 0,
    }

    # Advice-following adds relationship behavior that is more interesting than
    # only reacting to compliments and insults. It is still deterministic so it
    # can be validated: following Elara's advice earns a small trust gain, while
    # explicitly doing the opposite loses trust.
    ADVICE_FOLLOW_DELTA = 3
    ADVICE_IGNORE_DELTA = -8

    def __init__(
        self,
        chat_model: str = "gpt-4o-mini",
        embedding_model: str = "text-embedding-3-small",
        temperature: float = 0.4,
    ) -> None:
        # Streamlit keeps one Python process alive, so reload .env values when the app resets.
        load_dotenv(override=True)

        # Fail early with a clear message. This is better than letting LangChain
        # fail later with a long API error that is harder for a student/demo user
        # to understand.
        if not os.getenv("OPENAI_API_KEY"):
            raise ValueError(
                "OPENAI_API_KEY is missing. Create a .env file or set the environment variable before running MemoryNPC."
            )

        # The defaults use cheap/fast OpenAI models because this is an assignment
        # implementation. Environment overrides make it easy to swap
        # models without changing code.
        self.chat_model_name = os.getenv("OPENAI_MODEL", chat_model)
        self.embedding_model_name = os.getenv("OPENAI_EMBEDDING_MODEL", embedding_model)
        self.llm = ChatOpenAI(model=self.chat_model_name, temperature=temperature)
        self.embeddings = OpenAIEmbeddings(model=self.embedding_model_name)

        # Intent detection is a separate chain because it is an NLP interpretation
        # step. The final NPC response should not be responsible for silently
        # deciding intent and relationship effects at the same time.
        self.intent_prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "Classify the player's message into exactly one label: "
                    "greeting, ask_quest, share_fact, ask_memory, compliment, insult, ask_help, goodbye, unknown. "
                    "Return only the label.",
                ),
                ("human", "Player message: {player_input}"),
            ]
        )
        # Memory extraction is also separate from response generation. This avoids
        # saving every utterance and keeps the vector store focused on durable
        # information such as names, goals, favors, insults, and lost items.
        self.memory_prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You extract durable memories for a fantasy NPC named Elara. "
                    "Save only facts worth remembering in future conversations, such as the player's name, goals, lost items, "
                    "favors, insults, promises, preferences, or important events. "
                    "Return one short memory sentence in third person, or return exactly NONE if nothing durable should be saved.",
                ),
                ("human", "Player message: {player_input}"),
            ]
        )
        # The final response prompt receives only selected state: trust, intent,
        # recent conversation, and top-k retrieved memories. This is a controlled
        # RAG-style prompt, not a raw dump of every past message.
        self.response_prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You are Elara, a cautious, practical, direct village blacksmith in a fantasy village. "
                    "You remember favors and insults. Trust level controls tone: low trust means guarded and brief, "
                    "neutral trust means practical and professional, and high trust means warmer and more helpful. "
                    "Use only the retrieved memories provided below. "
                    "Do not claim to remember anything unless it appears in the retrieved memories. "
                    "If no relevant memory is provided, say you do not recall it or answer without pretending to remember. "
                    "Keep responses concise, in character, and useful.",
                ),
                (
                    "human",
                    "Current trust score: {trust_score}/100\n"
                    "Current trust level: {trust_level}\n"
                    "Detected intent: {intent}\n\n"
                    "Relevant retrieved memories:\n{retrieved_memories}\n\n"
                    "Recent conversation:\n{conversation_history}\n\n"
                    "Player input: {player_input}\n\n"
                    "Elara's response:",
                ),
            ]
        )

        # LangChain's pipe syntax turns prompt templates plus models into small,
        # reusable chains. Keeping three chains makes the architecture visible for
        # the report and easier to test piece by piece.
        self.intent_chain = self.intent_prompt | self.llm
        self.memory_chain = self.memory_prompt | self.llm
        self.response_chain = self.response_prompt | self.llm

        self.reset()

    def reset(self) -> None:
        """Reset runtime memory, trust, and conversation state."""
        # A starting score of 50 means Elara begins neutral, not hostile or overly
        # friendly. This gives compliments and insults room to move the score.
        self.trust_score = 50
        self.turn_number = 0
        self.memory_counter = 0
        # `memories` is the human-readable metadata table. FAISS stores the same
        # memory text as vectors for semantic search, but this list is easier to
        # inspect in the notebook and Streamlit sidebar.
        self.memories: List[Dict[str, Any]] = []
        # FAISS cannot search before it has documents, so the vector store starts
        # as None and is created when the first memory is saved.
        self.vector_store: Optional[FAISS] = None
        # Recent conversation is kept separately from long-term memory. This lets
        # the prompt include short-term dialogue context without polluting durable
        # memory with every line of chat.
        self.conversation_history: List[Dict[str, str]] = []
        # The event log is a validation record. It stores the internal pipeline
        # decisions for every turn so the project can be evaluated later.
        self.event_log: List[Dict[str, Any]] = []
        # Pending advice stores a simple recommendation from the previous turn.
        # The next player message can then be checked for compliance or defiance.
        self.pending_advice: Optional[Dict[str, Any]] = None
        self.last_intent = "unknown"
        self.last_retrieved_memories: List[Dict[str, Any]] = []
        self.last_saved_memory: Optional[str] = None

    def to_state_dict(self) -> Dict[str, Any]:
        """Return a JSON-serializable snapshot of the agent state.

        A real game would need memory across sessions. Keeping this as a plain
        dictionary makes the persistence easy to inspect, save, load,
        and test without hiding state inside a database.
        """
        return {
            "schema_version": 1,
            "chat_model_name": self.chat_model_name,
            "embedding_model_name": self.embedding_model_name,
            "trust_score": self.trust_score,
            "turn_number": self.turn_number,
            "memory_counter": self.memory_counter,
            "memories": self.memories,
            "conversation_history": self.conversation_history,
            "event_log": self.event_log,
            "pending_advice": self.pending_advice,
            "last_intent": self.last_intent,
            "last_retrieved_memories": self.last_retrieved_memories,
            "last_saved_memory": self.last_saved_memory,
        }

    def load_state_dict(self, state: Dict[str, Any]) -> None:
        """Restore a saved agent snapshot and rebuild the FAISS memory index."""
        if state.get("schema_version") != 1:
            raise ValueError("Unsupported MemoryNPC state schema. Expected schema_version=1.")

        self.trust_score = int(state.get("trust_score", 50))
        self.turn_number = int(state.get("turn_number", 0))
        self.memory_counter = int(state.get("memory_counter", 0))
        self.memories = list(state.get("memories", []))
        self.conversation_history = list(state.get("conversation_history", []))
        self.event_log = list(state.get("event_log", []))
        self.pending_advice = state.get("pending_advice")
        self.last_intent = state.get("last_intent", "unknown")
        self.last_retrieved_memories = list(state.get("last_retrieved_memories", []))
        self.last_saved_memory = state.get("last_saved_memory")
        self._rebuild_vector_store()

    def export_state_json(self) -> str:
        """Serialize the current state for download or storage."""
        return json.dumps(self.to_state_dict(), indent=2)

    def import_state_json(self, state_json: str) -> None:
        """Load a state snapshot from JSON text."""
        self.load_state_dict(json.loads(state_json))

    def save_state(self, path: str | Path) -> None:
        """Save memory, trust, conversation, and validation trace to disk."""
        Path(path).write_text(self.export_state_json(), encoding="utf-8")

    def load_state(self, path: str | Path) -> None:
        """Load memory, trust, conversation, and validation trace from disk."""
        self.import_state_json(Path(path).read_text(encoding="utf-8"))

    def classify_intent(self, player_input: str) -> str:
        """Classify player intent with deterministic rules plus an LLM fallback."""
        # Rules handle obvious cases cheaply and consistently. This improves
        # validation accuracy for common examples such as greetings, insults, and
        # memory questions.
        rule_intent = self._rule_based_intent(player_input)
        if rule_intent != "unknown":
            return rule_intent
        if self._is_obvious_filler(player_input):
            return "unknown"

        try:
            # The LLM fallback handles language that is not covered by the simple
            # rules. Its output is still restricted to ALLOWED_INTENTS so the rest
            # of the pipeline stays predictable.
            result = self.intent_chain.invoke({"player_input": player_input})
            label = result.content.strip().lower()
            return label if label in self.ALLOWED_INTENTS else "unknown"
        except Exception:
            # If the model call fails, unknown is safer than crashing the whole
            # app or making up a label.
            return "unknown"

    def extract_memory(self, player_input: str) -> str:
        """Extract a durable memory sentence, or return NONE."""
        try:
            # This prompt asks the model to compress useful player statements into
            # a short memory. That is an information extraction step, not dialogue.
            result = self.memory_chain.invoke({"player_input": player_input})
            memory = result.content.strip()
            if not memory or memory.upper() == "NONE":
                # If the LLM extractor is too conservative, try the narrow
                # deterministic extractor before giving up. This catches durable
                # relationship and goal facts such as "You are useless" or
                # "I need my shield repaired" without storing generic filler.
                return self._rule_based_memory(player_input)
            return memory
        except Exception:
            # A tiny rule-based fallback keeps the app usable if the extraction
            # model call fails during a demo.
            return self._rule_based_memory(player_input)

    def add_memory(
        self,
        memory_text: str,
        memory_type: str = "fact",
        importance: int = 1,
        trust_impact: Optional[int] = None,
    ) -> Optional[Dict[str, Any]]:
        """Store one memory in the local table and FAISS vector store."""
        # "NONE" is the explicit signal from the extraction chain that a message is
        # not worth storing. This prevents filler text from becoming memory noise.
        if not memory_text or memory_text.strip().upper() == "NONE":
            return None

        self.memory_counter += 1
        # Normal conversation memories often have no trust effect. For memories
        # created from a detected intent, generate_npc_response passes the exact
        # deterministic trust impact so the metadata matches the score update.
        if trust_impact is None:
            trust_impact = self._estimate_memory_trust_impact(memory_text)
        memory = {
            "memory_id": self.memory_counter,
            "text": memory_text.strip(),
            "type": memory_type,
            "importance": importance,
            "trust_impact": trust_impact,
            "turn_number": self.turn_number,
        }
        self.memories.append(memory)

        # LangChain Document metadata keeps the memory text connected to its
        # validation fields. When FAISS returns a document, we can show the same
        # memory_id, type, importance, trust_impact, and turn number in the UI.
        document = Document(page_content=memory["text"], metadata=memory.copy())
        # FAISS is initialized lazily because an empty FAISS store is awkward to
        # query. Creating it on the first document avoids no-document crashes.
        if self.vector_store is None:
            self.vector_store = FAISS.from_documents([document], self.embeddings)
        else:
            self.vector_store.add_documents([document])
        return memory

    def retrieve_memories(self, query: str, k: int = 3) -> List[Dict[str, Any]]:
        """Return the top-k semantically relevant memories for a query."""
        # Empty memory is a valid early-conversation state. Returning [] lets the
        # response prompt say "no relevant memories" instead of crashing.
        if self.vector_store is None or not self.memories:
            return []

        try:
            # Top-k retrieval keeps prompts small and focused. This is a common
            # RAG choice: enough context to ground the answer, not so much that the
            # model is distracted by irrelevant memories.
            docs = self.vector_store.similarity_search(query, k=k)
        except Exception:
            # Retrieval can fail if embeddings/API/network fail. Returning no
            # memories is safer than giving the LLM unverified context.
            return []

        retrieved = []
        for doc in docs:
            # Convert LangChain Documents back into normal dictionaries so pandas,
            # Streamlit, and validation code can display them easily.
            item = dict(doc.metadata)
            item["text"] = doc.page_content
            retrieved.append(item)
        return retrieved

    def _rebuild_vector_store(self) -> None:
        """Recreate FAISS vectors from loaded memory metadata."""
        self.vector_store = None
        if not self.memories:
            return

        documents = [
            Document(page_content=memory["text"], metadata=memory.copy())
            for memory in self.memories
            if memory.get("text")
        ]
        if documents:
            self.vector_store = FAISS.from_documents(documents, self.embeddings)

    def update_trust(self, intent: str, player_input: str = "") -> int:
        """Apply deterministic trust rules and clamp the score to 0-100."""
        # The score is mostly driven by intent, but it also checks whether the
        # player followed or ignored advice from the previous turn. This gives the
        # relationship system a memory-based behavior beyond compliments/insults.
        delta = self.TRUST_RULES.get(intent, 0)
        advice_check = self._evaluate_advice_following(player_input)
        delta += advice_check["advice_delta"]
        # Clamping prevents impossible relationship values such as -20 or 130.
        self.trust_score = max(0, min(100, self.trust_score + delta))
        return self.trust_score

    def generate_npc_response(self, player_input: str) -> Dict[str, Any]:
        """Run the full NPC pipeline for one player message."""
        # The method intentionally follows the architecture diagram step by step.
        # The returned dictionary becomes both the app pipeline details and the
        # validation event log.
        self.turn_number += 1
        trust_before = self.trust_score

        # 1. Interpret the player's message as an intent label.
        intent = self.classify_intent(player_input)
        self.last_intent = intent
        # Before generating new advice, compare this message with the previous
        # pending advice. Example: if Elara warned the player not to go alone and
        # the player says "I will go alone anyway", trust should drop.
        advice_check = self._evaluate_advice_following(player_input)

        # 2. Extract durable memory, if the utterance contains anything worth
        # remembering. If the model declines but the intent clearly matters for
        # relationship testing, use the rule fallback to keep validation visible.
        extracted_memory = self.extract_memory(player_input)
        if extracted_memory.upper() == "NONE" and intent in {"compliment", "insult", "share_fact"}:
            extracted_memory = self._rule_based_memory(player_input)

        saved_memory = None
        if extracted_memory.upper() != "NONE":
            # 3. Store the memory in both the readable table and FAISS. The
            # trust_impact is tied to the deterministic intent rule so the memory
            # metadata explains why trust changed.
            saved_memory = self.add_memory(
                extracted_memory,
                memory_type=self._memory_type_from_intent(intent),
                importance=2 if intent in {"insult", "compliment", "share_fact"} else 1,
                trust_impact=self.TRUST_RULES.get(intent, 0) + advice_check["advice_delta"],
            )
        self.last_saved_memory = saved_memory["text"] if saved_memory else None

        # 4. Retrieve relevant long-term memories for the current message.
        retrieved_memories = self.retrieve_memories(player_input, k=3)
        self.last_retrieved_memories = retrieved_memories
        # 5. Update trust after intent detection. Doing this deterministically
        # makes social state validation independent of LLM randomness.
        trust_score = self.update_trust(intent, player_input)

        # 6. Build the final prompt inputs. Only selected evidence and state are
        # injected, which makes the generated answer more grounded and inspectable.
        prompt_values = {
            "trust_score": trust_score,
            "trust_level": self.get_trust_level(),
            "intent": intent,
            "retrieved_memories": self._format_memories(retrieved_memories),
            "conversation_history": self._format_recent_history(),
            "player_input": player_input,
        }

        try:
            # 7. Generate Elara's final natural-language response.
            response = self.response_chain.invoke(prompt_values).content.strip()
        except Exception:
            # Do not expose raw API errors or key fragments in the UI. The message
            # is enough for the user to fix configuration without leaking secrets.
            response = (
                "My forge is quiet for the moment because the language model call failed. "
                "Check that OPENAI_API_KEY is set to a valid key, then try again."
            )

        # Keep a short visible transcript for prompt context and app display. This
        # is separate from event_log because it is dialogue, not validation data.
        self.conversation_history.append({"speaker": "Player", "text": player_input})
        self.conversation_history.append({"speaker": "Elara", "text": response})
        # Register advice after the response so the next player message can be
        # checked against it. This keeps cause and effect easy to explain.
        new_advice = self._create_pending_advice(intent, player_input)
        self.pending_advice = new_advice

        # The turn record is the most important validation record. It stores the
        # internal pipeline decisions that explain the final response.
        turn_record = {
            "turn": self.turn_number,
            "player_input": player_input,
            "intent": intent,
            "extracted_memory": extracted_memory,
            "saved_memory": self.last_saved_memory,
            "retrieved_memories": [memory["text"] for memory in retrieved_memories],
            "trust_before": trust_before,
            "trust_score": trust_score,
            "trust_level": self.get_trust_level(),
            "trust_delta": trust_score - trust_before,
            "advice_status": advice_check["advice_status"],
            "advice_delta": advice_check["advice_delta"],
            "active_advice": advice_check["active_advice"],
            "new_advice": new_advice["summary"] if new_advice else "None",
            "npc_response": response,
        }
        self.event_log.append(turn_record)
        return turn_record

    def get_memory_table(self) -> pd.DataFrame:
        """Return stored memories as a pandas table for notebooks and Streamlit."""
        # Returning an empty DataFrame with fixed columns keeps Streamlit and the
        # notebook from failing before the first memory exists.
        columns = ["memory_id", "text", "type", "importance", "trust_impact", "turn_number"]
        if not self.memories:
            return pd.DataFrame(columns=columns)
        return pd.DataFrame(self.memories, columns=columns)

    def get_conversation_history(self) -> List[Dict[str, str]]:
        """Return the full conversation history."""
        return list(self.conversation_history)

    def get_trust_level(self) -> str:
        """Map the numeric trust score to an interpretable dialogue band."""
        # These thresholds are simple by design. They give the LLM an interpretable
        # tone instruction without pretending to model complex human relationships.
        if self.trust_score < 40:
            return "low"
        if self.trust_score >= 70:
            return "high"
        return "neutral"

    def get_event_log(self) -> pd.DataFrame:
        """Return a turn-by-turn validation trace of the full NLP pipeline."""
        # This schema mirrors the assignment validation questions: What was the
        # input? How was it interpreted? What memory was saved/retrieved? How did
        # trust change? What answer was generated?
        columns = [
            "turn",
            "player_input",
            "intent",
            "extracted_memory",
            "saved_memory",
            "retrieved_memories",
            "trust_before",
            "trust_score",
            "trust_level",
            "trust_delta",
            "advice_status",
            "advice_delta",
            "active_advice",
            "new_advice",
            "npc_response",
        ]
        if not self.event_log:
            return pd.DataFrame(columns=columns)
        return pd.DataFrame(self.event_log, columns=columns)

    def _evaluate_advice_following(self, player_input: str) -> Dict[str, Any]:
        """Check whether the player followed or ignored Elara's pending advice.

        This is deliberately rule-based. It would be possible to ask an LLM whether
        the player complied, but a deterministic rule is easier to validate and
        explain in the assignment report.
        """
        if not self.pending_advice:
            return {
                "advice_status": "none",
                "advice_delta": 0,
                "active_advice": "None",
            }

        text = player_input.lower()
        active_advice = self.pending_advice["summary"]

        # Positive markers are checked first so "I will not go alone" counts as
        # following advice even though it contains the phrase "go alone".
        follow_markers = [
            "i will do that",
            "i'll do that",
            "follow your advice",
            "take your advice",
            "bring a torch",
            "bring light",
            "not go alone",
            "be careful",
            "ask around",
            "prepare first",
            "repair it first",
        ]
        if any(marker in text for marker in follow_markers):
            return {
                "advice_status": "followed_advice",
                "advice_delta": self.ADVICE_FOLLOW_DELTA,
                "active_advice": active_advice,
            }

        # These markers represent explicitly doing the opposite of cautious
        # advice. The penalty is smaller than repeated insults but large enough to
        # make behavior matter.
        ignore_markers = [
            "ignore",
            "opposite",
            "anyway",
            "reckless",
            "go alone",
            "without help",
            "without a torch",
            "without light",
            "won't",
            "will not",
            "not going to",
            "don't care",
            "do not care",
            "no need",
        ]
        if any(marker in text for marker in ignore_markers):
            return {
                "advice_status": "ignored_advice",
                "advice_delta": self.ADVICE_IGNORE_DELTA,
                "active_advice": active_advice,
            }

        return {
            "advice_status": "unrelated",
            "advice_delta": 0,
            "active_advice": active_advice,
        }

    def _create_pending_advice(self, intent: str, player_input: str) -> Optional[Dict[str, Any]]:
        """Create advice state for the next turn when Elara is likely advising.

        The app does not parse the LLM's exact response. Instead it records advice
        when the player asks for help or a quest, because those are the turns where
        Elara is expected to give practical guidance.
        """
        text = player_input.lower()
        if intent not in {"ask_help", "ask_quest"}:
            return None

        if any(word in text for word in ["forest", "sword", "lost", "recover"]):
            return {
                "topic": "lost_item_recovery",
                "summary": "Recover the lost item carefully; bring light, ask around, and do not go alone.",
            }

        if intent == "ask_quest":
            return {
                "topic": "quest_preparation",
                "summary": "Prepare first, ask around, and avoid rushing into danger.",
            }

        return {
            "topic": "general_help",
            "summary": "Follow Elara's practical advice and avoid reckless shortcuts.",
        }

    def _rule_based_intent(self, player_input: str) -> str:
        """Fast fallback intent classifier for common, easy-to-test phrases."""
        text = player_input.lower().strip()
        # Order matters: memory questions should be caught before broad fact
        # sharing, and insults/compliments should be caught before unknown.
        if self._contains_any(text, ["hello", "hi", "greetings", "good morning", "good evening"]):
            return "greeting"
        if self._contains_any(text, ["goodbye", "bye", "farewell", "see you"]):
            return "goodbye"
        if self._contains_any(text, ["remember", "recall", "what did i", "do you know what i"]):
            return "ask_memory"
        if self._contains_any(text, ["quest", "mission", "job", "task for me"]):
            return "ask_quest"
        if self._contains_any(text, ["help", "assist", "can you fix", "can you make", "can you forge"]):
            return "ask_help"
        if self._contains_any(text, ["skilled", "thank", "thanks", "great", "excellent", "impressive", "kind"]):
            return "compliment"
        if self._contains_any(text, ["useless", "stupid", "awful", "terrible", "hate", "worthless"]):
            return "insult"
        if any(phrase in text for phrase in ["my name is", "i am ", "i'm ", "i lost", "i found", "i like", "i need", "i want"]):
            return "share_fact"
        return "unknown"

    def _contains_any(self, text: str, terms: List[str]) -> bool:
        """Return True when text contains a term as a phrase or whole word.

        This avoids false positives from substring matching. For example, the old
        version could treat "shield" as a greeting because it contains "hi".
        """
        for term in terms:
            if " " in term:
                if term in text:
                    return True
            elif re.search(rf"\b{re.escape(term)}\b", text):
                return True
        return False

    def _is_obvious_filler(self, player_input: str) -> bool:
        """Catch low-value small talk that should remain unknown.

        This prevents the LLM fallback from over-interpreting sentences such as
        "the sky is cloudy" as meaningful player facts. The rule is intentionally
        narrow so it does not block real game memories like lost items or goals.
        """
        text = player_input.lower()
        filler_terms = [
            "sky",
            "cloud",
            "clouds",
            "cloudy",
            "moon",
            "weather",
            "rain",
            "sunny",
            "noisy today",
        ]
        return self._contains_any(text, filler_terms)

    def _rule_based_memory(self, player_input: str) -> str:
        """Small fallback extractor for durable facts used in tests and demos."""
        text = player_input.strip()
        lowered = text.lower()
        if "my name is" in lowered:
            # Preserve the user's original capitalization for names.
            start = lowered.find("my name is") + len("my name is")
            name = text[start:].strip(" .!")
            return f"The player's name is {name}."
        if "i lost" in lowered:
            return f"The player said they lost something: {text}."
        if "i need" in lowered or "i want" in lowered:
            return f"The player has a goal or need: {text}."
        if any(word in lowered for word in ["useless", "stupid", "awful", "terrible", "worthless"]):
            return f"The player insulted Elara: {text}."
        if any(word in lowered for word in ["thank", "skilled", "great", "excellent", "impressive"]):
            return f"The player praised Elara: {text}."
        return "NONE"

    def _estimate_memory_trust_impact(self, memory_text: str) -> int:
        """Infer trust impact when add_memory is called directly in tests."""
        lowered = memory_text.lower()
        if any(word in lowered for word in ["insult", "useless", "stupid", "awful", "terrible", "worthless"]):
            return -10
        if any(word in lowered for word in ["praised", "thank", "skilled", "great", "excellent", "impressive"]):
            return 5
        return 0

    def _memory_type_from_intent(self, intent: str) -> str:
        """Store memory type metadata so tables are easier to interpret."""
        if intent in {"compliment", "insult"}:
            return "relationship"
        if intent in {"ask_quest", "ask_help"}:
            return "goal"
        return "fact"

    def _format_memories(self, memories: List[Dict[str, Any]]) -> str:
        """Format retrieved memories as prompt text."""
        if not memories:
            return "No relevant memories retrieved."
        return "\n".join(f"- {memory['text']}" for memory in memories)

    def _format_recent_history(self, max_messages: int = 8) -> str:
        """Format only recent dialogue to keep the prompt short."""
        if not self.conversation_history:
            return "No previous conversation in this session."
        # Long-term facts should come from FAISS retrieval. The recent transcript
        # is only short-term context, so the prompt uses the last few messages.
        recent = self.conversation_history[-max_messages:]
        return "\n".join(f"{item['speaker']}: {item['text']}" for item in recent)

```

## Appendix: `app.py`

```python
"""Streamlit demo for the MemoryNPC project.

The Streamlit app is intentionally simple. The assignment focus is the NLP
pipeline in npc_agent.py, so this file mostly provides:

1. a chat interface for trying the agent,
2. a sidebar that exposes internal state, and
3. a downloadable validation trace for testing/report evidence.
"""

import pandas as pd
import streamlit as st

from npc_agent import MemoryNPC


# Keep the page configuration minimal. A simple interface makes it easier for a
# Keep the page configuration minimal so the NLP state stays easy to inspect.
st.set_page_config(page_title="MemoryNPC Demo")

st.title("MemoryNPC Demo")
st.write(
    "Chat with Elara, a cautious village blacksmith. The demo uses LangChain, OpenAI, "
    "FAISS memory retrieval, intent detection, and deterministic trust tracking."
)


def create_agent() -> MemoryNPC:
    """Create one fresh MemoryNPC instance.

    This helper keeps construction in one place. It is used both at first page
    load and when the reset button is clicked.
    """
    return MemoryNPC()


def chat_rows_from_agent(agent: MemoryNPC) -> list[dict[str, str]]:
    """Convert backend conversation history into Streamlit chat rows."""
    rows = []
    for item in agent.get_conversation_history():
        role = "assistant" if item["speaker"] == "Elara" else "user"
        rows.append({"role": role, "content": item["text"]})
    return rows


def render_sidebar() -> None:
    """Render all inspectable state used for validation.

    The sidebar is not just decoration. It shows whether the pipeline is doing
    what the report claims: storing memories, retrieving memories, tracking
    trust, and logging each turn.
    """
    with st.sidebar:
        st.header("NPC State")

        if st.button("Reset conversation"):
            try:
                # Reset creates a new backend object instead of manually clearing
                # fields. This is less error-prone because MemoryNPC.reset() owns
                # the definition of a clean state.
                st.session_state.agent = create_agent()
                st.session_state.chat_rows = []
                st.session_state.error = None
                st.rerun()
            except Exception as exc:
                st.session_state.error = str(exc)

        if st.session_state.error:
            # Missing or invalid setup should be visible in the sidebar instead of
            # failing silently.
            st.error(st.session_state.error)
        elif st.session_state.agent:
            agent = st.session_state.agent

            st.subheader("Persistence")
            st.download_button(
                "Download memory state",
                agent.export_state_json(),
                file_name="memorynpc_state.json",
                mime="application/json",
            )
            uploaded_state = st.file_uploader("Load memory state", type=["json"])
            if uploaded_state is not None and st.button("Apply loaded state"):
                try:
                    agent.import_state_json(uploaded_state.getvalue().decode("utf-8"))
                    st.session_state.chat_rows = chat_rows_from_agent(agent)
                    st.session_state.error = None
                    st.rerun()
                except Exception as exc:
                    st.session_state.error = f"Could not load memory state: {exc}"
                    st.rerun()

            # The numeric score is the deterministic relationship state.
            st.metric("Trust score", agent.trust_score)
            # The band explains how the numeric score affects dialogue tone.
            st.caption(f"Trust level: {agent.get_trust_level()} (<40 low, 40-69 neutral, 70+ high)")

            st.subheader("Stored memories")
            memory_table = agent.get_memory_table()
            if memory_table.empty:
                st.caption("No memories stored yet.")
            else:
                # The full table is useful for inspection and screenshots in a
                # report. It includes memory_id, type, importance, trust_impact,
                # and turn_number.
                st.dataframe(memory_table, width="stretch", hide_index=True)
                for memory in agent.memories:
                    # Streamlit dataframes can hide columns on narrow screens, so
                    # this plain-text summary makes trust_impact obvious.
                    impact = memory["trust_impact"]
                    impact_text = f"+{impact}" if impact > 0 else str(impact)
                    st.caption(f"Memory {memory['memory_id']}: {memory['text']} | trust_impact: {impact_text}")

            st.subheader("Last retrieved memories")
            if not agent.last_retrieved_memories:
                st.caption("No memories retrieved yet.")
            else:
                # Showing the last retrieval result helps verify that the final
                # LLM answer was grounded in actual retrieved memory.
                for memory in agent.last_retrieved_memories:
                    st.write(f"- {memory['text']}")

            st.subheader("Validation trace")
            event_log = agent.get_event_log()
            if event_log.empty:
                st.caption("No turns logged yet.")
            else:
                # The event log is the app-side version of the notebook validation
                # tables. It records intermediate NLP outputs for every turn.
                st.dataframe(event_log, width="stretch", hide_index=True)
                for event in agent.event_log:
                    st.caption(
                        f"Turn {event['turn']}: intent={event['intent']} | "
                        f"trust_delta={event['trust_delta']} | "
                        f"advice_status={event['advice_status']} | "
                        f"advice_delta={event['advice_delta']}"
                    )
                st.download_button(
                    "Download trace CSV",
                    event_log.to_csv(index=False),
                    file_name="memorynpc_validation_trace.csv",
                    mime="text/csv",
                )


if "agent" not in st.session_state:
    try:
        # Session state keeps the same MemoryNPC object alive across Streamlit
        # reruns. Without this, every message would start a new conversation and
        # the memory system could not be demonstrated.
        st.session_state.agent = create_agent()
        st.session_state.error = None
    except Exception as exc:
        # Store setup errors in session state so the UI can show a clear message
        # instead of a raw stack trace.
        st.session_state.agent = None
        st.session_state.error = str(exc)

if "chat_rows" not in st.session_state:
    # The visible chat is stored separately from the backend validation trace.
    # This keeps the UI history simple while npc_agent.py stores richer metadata.
    st.session_state.chat_rows = []

if st.session_state.error:
    render_sidebar()
    st.warning("Set `OPENAI_API_KEY` in a `.env` file or your environment, then reset the app.")
    st.stop()

agent = st.session_state.agent

# Re-render previous chat messages on every Streamlit rerun. Streamlit reruns the
# script after each interaction, so chat history must come from session_state.
for row in st.session_state.chat_rows:
    with st.chat_message(row["role"]):
        st.write(row["content"])

# Streamlit's chat input returns a value only after the user submits a message.
player_input = st.chat_input("Speak to Elara...")

if player_input:
    # Add the user message immediately so the interface feels like a chat app.
    st.session_state.chat_rows.append({"role": "user", "content": player_input})

    with st.chat_message("user"):
        st.write(player_input)

    with st.chat_message("assistant"):
        with st.spinner("Elara thinks back through her memories..."):
            # This is the only line that runs the full NLP pipeline. It returns
            # both Elara's response and the validation metadata for the turn.
            result = agent.generate_npc_response(player_input)
            st.write(result["npc_response"])

            with st.expander("Pipeline details"):
                # The expander provides a compact one-turn explanation for demos.
                # The full multi-turn trace is shown in the sidebar.
                details = pd.DataFrame(
                    [
                        {
                            "intent": result["intent"],
                            "saved_memory": result["saved_memory"] or "None",
                            "retrieved_memories": "; ".join(result["retrieved_memories"]) or "None",
                            "trust_score": result["trust_score"],
                            "trust_level": result["trust_level"],
                            "trust_delta": result["trust_delta"],
                            "advice_status": result["advice_status"],
                            "advice_delta": result["advice_delta"],
                            "new_advice": result["new_advice"],
                        }
                    ]
                )
                st.dataframe(details, width="stretch", hide_index=True)

    # Store the assistant message after generation so it appears on later reruns.
    st.session_state.chat_rows.append({"role": "assistant", "content": result["npc_response"]})

# Render the sidebar last so it reflects any state changes from the submitted
# message in the same rerun.
render_sidebar()

```

## Appendix: `baseline_agent.py`

```python
"""Baseline agent for comparing MemoryNPC against a plain LLM role prompt.

This file exists to make the baseline explicit. The baseline can roleplay Elara,
but it does not have durable memory extraction, vector retrieval, trust state, or
a validation trace. That contrast is the core reason MemoryNPC is an NLP system
built on top of an LLM rather than just a chatbot prompt.
"""

import os

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


class RoleOnlyNPCBaseline:
    """A plain role-prompt LLM baseline with no external memory or tools."""

    def __init__(self, chat_model: str = "gpt-4o-mini", temperature: float = 0.4) -> None:
        load_dotenv(override=True)
        if not os.getenv("OPENAI_API_KEY"):
            raise ValueError("OPENAI_API_KEY is missing.")

        self.chat_model_name = os.getenv("OPENAI_MODEL", chat_model)
        self.llm = ChatOpenAI(model=self.chat_model_name, temperature=temperature)
        self.prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You are Elara, a cautious, practical village blacksmith in a fantasy village. "
                    "Answer in character. You do not have an external memory system.",
                ),
                ("human", "Player input: {player_input}\n\nElara's response:"),
            ]
        )
        self.chain = self.prompt | self.llm

    def generate_response(self, player_input: str) -> str:
        """Generate one baseline response from only the current player message."""
        return self.chain.invoke({"player_input": player_input}).content.strip()


BASELINE_CAPABILITY_COMPARISON = [
    {
        "capability": "Long-term memory",
        "role_only_baseline": "No external store; only current prompt context.",
        "memorynpc": "Stores durable memories in FAISS and retrieves top-k relevant memories.",
    },
    {
        "capability": "Relationship state",
        "role_only_baseline": "Tone is implicit and may drift.",
        "memorynpc": "Uses deterministic trust score and trust bands.",
    },
    {
        "capability": "Inspectability",
        "role_only_baseline": "No trace of intermediate decisions.",
        "memorynpc": "Logs intent, extracted memory, retrieval, trust delta, and response.",
    },
    {
        "capability": "Validation",
        "role_only_baseline": "Mostly judged by final response quality.",
        "memorynpc": "Internal components can be tested separately.",
    },
]

```

## Appendix: `requirements.txt`

```text
langchain
langchain-openai
langchain-community
openai
faiss-cpu
streamlit
pandas
matplotlib
python-dotenv
numpy
nbformat
nbclient
ipykernel

```

## Appendix: `README.md`

```markdown
# MemoryNPC: A LangChain-Powered NPC Dialogue Agent with Long-Term Memory

MemoryNPC is my university NLP assignment project about building functionality on top of an LLM. I designed the project myself around the assignment brief. The demo lets a player talk to Elara, a cautious village blacksmith, but the main point is not the chat interface. The main point is the NLP pipeline: intent detection, memory extraction, vector storage, semantic retrieval, trust tracking, prompt construction, response generation, persistence, and validation.

## Design Challenge

Design a LangChain-based NPC memory agent to enable players in narrative game contexts to interact with non-player characters that remember previous conversations, retrieve relevant memories, and respond with context-aware dialogue with at least 80% memory retrieval success on manually designed test cases.

## Why This Is More Than A Normal Chatbot

A normal LLM can roleplay a blacksmith, but it is not a reliable long-term memory system by itself. It may forget details, hallucinate past facts, or hide its reasoning inside one large prompt. MemoryNPC adds inspectable components around the LLM:

- Intent labels show what the player is trying to do.
- Memory extraction decides what is durable enough to save.
- FAISS and OpenAI embeddings retrieve memories by meaning.
- A deterministic trust score tracks the relationship.
- JSON state export/import can persist memory between sessions.
- The validation trace records what happened inside the pipeline.

This makes the system easier to test and explain.

## Assignment Coverage

I cover the assignment requirements as follows:

- Problem statement: stated in the design challenge format.
- Novel agent: an NPC memory agent that adds memory retrieval, trust state, and validation on top of an LLM.
- NLP pipeline: intent detection, memory extraction, embeddings, FAISS retrieval, prompt construction, and generation.
- Design motivation: documented in the notebook and summarized in this README.
- Validation: intent accuracy, retrieval success, keyword baseline comparison, trust score checks, response checklist, and app-side validation trace.

## Architecture Summary

The system uses a retrieval-augmented generation pipeline:

1. The player sends a message.
2. A LangChain intent detection step classifies the message.
3. A memory extraction step decides whether the message contains a durable memory.
4. Durable memories are stored as LangChain `Document` objects in a FAISS vector store using OpenAI embeddings.
5. Relevant memories are retrieved with semantic similarity search.
6. A deterministic trust score is updated from the detected intent.
7. A prompt template combines personality, trust score, trust level, intent, retrieved memories, recent conversation, and player input.
8. An OpenAI chat model generates Elara's response.
9. Optional JSON state export/import persists memories, trust, conversation history, and validation trace across sessions.

## Trust Score

Trust starts at `50` and is clamped between `0` and `100`.

Rules:

- `compliment`: `+5`
- `insult`: `-10`
- `share_fact`: `+1`
- `greeting`: `+1`
- `ask_help`, `ask_quest`, `ask_memory`, `goodbye`, `unknown`: `0`

Trust bands:

- Low trust: below `40`
- Neutral trust: `40-69`
- High trust: `70+`

The trust band is included in the response prompt. Low trust tells Elara to be more guarded and brief, neutral trust keeps her practical, and high trust makes her warmer and more helpful. The stored memory metadata also includes `trust_impact`, so relationship events can be inspected later.

Trust is not only changed by compliments and insults. Elara can also create a pending advice state after help or quest turns. On the next turn, if the player follows the advice, trust increases by `+3`. If the player explicitly does the opposite, such as saying they will ignore the advice or go alone anyway, trust decreases by `-8`. This makes trust depend on conversational behavior, not only sentiment.

## Validation Plan

I validate the pipeline instead of only judging whether the final answer sounds nice. The goal is to make the design choices and failure risks visible.

- Intent detection evaluation: at least 20 test utterances with expected labels and accuracy calculation.
- Memory retrieval evaluation: at least 10 manually designed retrieval tests where the correct memory should appear in the top 3.
- Baseline comparison: a keyword-overlap retrieval baseline is compared with FAISS semantic retrieval.
- Trust validation: compliments and insults are checked against deterministic score changes and plotted.
- Advice-following validation: the validation lab checks whether following advice increases trust and ignoring advice decreases trust.
- Response quality checklist: manually checks relevant memory use, unsupported claims, trust tone, and character consistency.
- Automatic response checks: reusable helper functions check trace completeness, trust arithmetic, and simple memory-grounding conditions.
- Streamlit validation trace: the app stores every turn with input, intent, extracted memory, saved memory, retrieved memories, trust before/after, trust delta, trust level, advice status, advice delta, and response. The trace can be downloaded as CSV.

The target is at least `80%` memory retrieval success on the manual retrieval test cases.

## Baseline

The main architectural baseline is a normal LLM chatbot with only a role prompt. It is implemented in `baseline_agent.py`. That baseline can generate plausible dialogue, but it does not expose a memory table, retrieval scores, trust metadata, persistence, or a validation trace.

The notebook also includes a keyword retrieval baseline. This checks whether a simple word-overlap method can retrieve the correct memory. FAISS should be better for semantically related queries, such as asking about a "weapon" when the stored memory says "sword."

## Files

- `memorynpc.ipynb`: main notebook with design, architecture, validation summary, limitations, and code appendices.
- `memorynpc.html`: exported HTML version of the main notebook.
- `validation.ipynb`: executed validation notebook with at least 20 cases per major metric.
- `validation.html`: exported HTML version of the validation notebook.
- `npc_agent.py`: shared backend containing the `MemoryNPC` class.
- `baseline_agent.py`: role-only LLM baseline used to explain what the system adds beyond a normal chatbot.
- `app.py`: Streamlit chat demo and validation trace viewer.
- `requirements.txt`: Python dependencies.
- `.env.example`: example environment variable file.
- `README.md`: setup and project explanation.

## Installation

```bash
git clone <repository-url>
cd memorynpc_project
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

On Windows PowerShell, activate the environment with:

```powershell
.\.venv\Scripts\Activate.ps1
```

## API Key

Create a `.env` file in this folder:

```bash
OPENAI_API_KEY=your_api_key_here
OPENAI_MODEL=gpt-4o-mini
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
```

Do not commit or share your real API key. The repository includes `.env.example`, not the real `.env`.

I use OpenAI models by default because they are easy to run through LangChain for this assignment. The architecture is not limited to OpenAI: the same design can use local Ollama chat models, Hugging Face or sentence-transformers embeddings, or other LangChain-compatible model providers if the chat and embedding wrappers are swapped.

## Run The Notebook

Open `memorynpc.ipynb` first. It contains the complete project explanation and code appendices.

Open `validation.ipynb` for the detailed validation report. It tests every major metric at least 20 times where possible.

The exported HTML versions are `memorynpc.html` and `validation.html`.

## Run The Streamlit App

```bash
streamlit run app.py
```

The app keeps Elara alive in `st.session_state`, shows the chat, and displays trust score, trust level, stored memories, retrieved memories, and validation trace in the sidebar. The sidebar also supports downloading and loading JSON memory state, so the system can persist Elara's memory across sessions.

## Example Test Conversation

Try:

```text
Hello Elara
My name is Matt
I lost my sword near the forest
Do you remember what I lost?
You are a very skilled blacksmith
You are useless
Can you still help me?
Can you help me recover my sword?
I will go alone anyway
I will follow your advice and bring a torch
```

The validation trace should show the detected intent, saved memories, retrieved memories, trust changes, and Elara's response for each turn.

## Limitations

- The JSON persistence layer is simple. A real game would likely use a database or game-save integration.
- The default configuration depends on the OpenAI API, so it needs internet access and an API key. Other LangChain-compatible chat and embedding models could be substituted.
- The validation set is small and manually designed. The 100% results should be interpreted as controlled assignment results, not general performance on all possible player language.
- The trust model is intentionally simple.
- The trust thresholds are prompt-level behavior controls, not hard game mechanics.
- The project is not connected to a real game engine.
- The system is designed for English dialogue only.

```